# TTS Experiment — Coqui XTTS v2

**WeatherSpeak PH** — Gemma 4 Hackathon

## Objective

Convert the 6 radio bulletin scripts generated in notebook 06 into MP3 audio files
using **Coqui XTTS v2** — one file per language per bulletin.

### Language mapping
| Script language | XTTS v2 language code | Rationale |
|---|---|---|
| English (`en`) | `en` | Native support |
| Tagalog (`tl`) | `es` (Spanish) | Best available phoneme approximation — Filipino shares Spanish's 5-vowel system and consonant inventory |
| Cebuano (`ceb`) | `es` (Spanish) | Same rationale as Tagalog |

> **Note — Future alternative:** The Coqui TTS model zoo includes a community-contributed
> Tagalog model (`tts_models/tl/...`). This could replace the Spanish-phoneme fallback
> for native phoneme accuracy, at the cost of a different (inconsistent) voice.
> Worth evaluating once the pipeline is stable.

### Output
6 MP3 files saved to `data/tts_output/` — ready for download or web playback.

### Modal note
Core functions (`preprocess_for_tts`, `synthesize_to_mp3`) are self-contained
with no notebook globals — ready to wrap in `@app.function` for Modal deployment.

In [ ]:
import re
import time
import numpy as np
from pathlib import Path
from pydub import AudioSegment
from TTS.api import TTS

# --- Paths ---
data_dir = Path("../data")
input_dir = data_dir / "radio_bulletins"
output_dir = data_dir / "tts_output"
output_dir.mkdir(exist_ok=True)

# --- Language mapping ---
LANGUAGE_MAP = {"en": "en", "tl": "es", "ceb": "es"}
SPEAKER = "Damien Black"
SAMPLE_RATE = 24_000  # XTTS v2 native sample rate

# --- Input files ---
STEMS = [
    "PAGASA_20-19W_Pepito_SWB#01",
    "PAGASA_22-TC02_Basyang_TCA#01",
]
LANGUAGES = ["en", "tl", "ceb"]

input_files = [
    input_dir / f"{stem}_radio_{lang}.md"
    for stem in STEMS
    for lang in LANGUAGES
]

missing = [f for f in input_files if not f.exists()]
if missing:
    print(f"⚠  Missing input files: {[f.name for f in missing]}")
else:
    print(f"✓ {len(input_files)} input files found")

print(f"✓ Output dir: {output_dir.absolute()}")

## 1. Text Preprocessing

Strip markdown syntax and insert pause cues at section boundaries.
Section headings become `...` + heading text — XTTS v2 reads the ellipsis
as a sentence-boundary pause.

In [ ]:
def preprocess_for_tts(markdown_text: str) -> str:
    """Strip markdown formatting and insert pause cues at section boundaries.

    Modal-ready: pure function, no external state.
    """
    text = markdown_text

    # Section headings → pause cue + heading as spoken sentence
    text = re.sub(r"^#{1,6}\s+(.+)$", r"\n...\n\1.\n", text, flags=re.MULTILINE)

    # Horizontal rules — must come BEFORE bold/italic removal to avoid *** corruption
    text = re.sub(r"^[-*_]{3,}$", "", text, flags=re.MULTILINE)

    # Bold and italic markers
    text = re.sub(r"\*{1,3}(.+?)\*{1,3}", r"\1", text)
    text = re.sub(r"_{1,3}(.+?)_{1,3}", r"\1", text)

    # Inline code
    text = re.sub(r"`(.+?)`", r"\1", text)

    # Blockquotes
    text = re.sub(r"^>\s+", "", text, flags=re.MULTILINE)

    # Collapse 3+ blank lines to 2
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# --- Inline test on one real bulletin ---
sample_md = (input_dir / "PAGASA_20-19W_Pepito_SWB#01_radio_en.md").read_text()
sample_plain = preprocess_for_tts(sample_md)

print("PREPROCESSING SAMPLE (first 600 chars of plain text)")
print("=" * 60)
print(sample_plain[:600])
print("...")
print(f"\nOriginal: {len(sample_md)} chars  →  Preprocessed: {len(sample_plain)} chars")

## 2. TTS Synthesis Functions

Two self-contained functions — Modal-ready (no notebook globals in signatures).
- `chunk_text`: splits long text into XTTS v2-safe segments on sentence boundaries
- `synthesize_to_mp3`: full pipeline from plain text → MP3, no intermediate WAV

In [ ]:
def chunk_text(text: str, max_chars: int = 200) -> list[str]:
    """Split text into segments ≤ max_chars on sentence boundaries.

    XTTS v2 is unreliable above ~200 chars per call.
    Modal-ready: pure function, no external state.
    """
    sentences = re.split(r"(?<=[.!?…])\s+", text)
    chunks: list[str] = []
    current = ""
    for sentence in sentences:
        if not sentence.strip():
            continue
        if len(current) + len(sentence) + 1 <= max_chars:
            current = (current + " " + sentence).strip()
        else:
            if current:
                chunks.append(current)
            # If a single sentence exceeds max_chars, keep it as-is
            current = sentence
    if current:
        chunks.append(current)
    return chunks


def synthesize_to_mp3(
    text: str,
    language: str,
    output_path: Path,
    tts: TTS,
    speaker: str = "Damien Black",
    sample_rate: int = 24_000,
) -> Path:
    """Synthesize plain text to MP3 using XTTS v2.

    Modal-ready: all inputs are primitive types + Path; no notebook globals.

    Args:
        text: Plain text (no markdown). Use preprocess_for_tts() first.
        language: XTTS v2 language code ('en' or 'es').
        output_path: Destination MP3 path.
        tts: Loaded TTS instance (load once, reuse for all files).
        speaker: XTTS v2 built-in speaker name.
        sample_rate: Audio sample rate in Hz (XTTS v2 native is 24000).

    Returns:
        output_path on success.
    """
    chunks = chunk_text(text)
    audio_arrays: list[np.ndarray] = []

    for chunk in chunks:
        wav = tts.tts(text=chunk, speaker=speaker, language=language)
        audio_arrays.append(np.array(wav, dtype=np.float32))

    # Concatenate all chunks in memory
    combined = np.concatenate(audio_arrays)

    # float32 [-1, 1] → int16 PCM for pydub
    pcm = (combined * 32_767).clip(-32_768, 32_767).astype(np.int16)

    segment = AudioSegment(
        pcm.tobytes(),
        frame_rate=sample_rate,
        sample_width=2,  # 16-bit = 2 bytes
        channels=1,
    )
    segment.export(str(output_path), format="mp3", bitrate="128k")
    return output_path


print("✓ chunk_text and synthesize_to_mp3 defined")

# Quick sanity check on chunking
sample_chunks = chunk_text(sample_plain)
print(f"  Sample bulletin: {len(sample_plain)} chars → {len(sample_chunks)} chunks")
print(f"  Longest chunk: {max(len(c) for c in sample_chunks)} chars")

## 3. Load Model + Batch Synthesis

XTTS v2 downloads ~1.8 GB on first run (cached after). Load once, reuse for all files.

In [ ]:
print("Loading XTTS v2 model (downloads ~1.8 GB on first run)...")
t0 = time.time()
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
print(f"✓ Model loaded in {time.time() - t0:.1f}s")

# Confirm speaker exists
available_speakers = tts.speakers or []
if SPEAKER not in available_speakers:
    print(f"⚠  '{SPEAKER}' not found. Available male voices:")
    print([s for s in available_speakers if any(c.isupper() for c in s)])
else:
    print(f"✓ Speaker '{SPEAKER}' confirmed")

In [ ]:
results = []

for stem in STEMS:
    for lang in LANGUAGES:
        input_path = input_dir / f"{stem}_radio_{lang}.md"
        output_path = output_dir / f"{stem}_radio_{lang}.mp3"
        xtts_lang = LANGUAGE_MAP[lang]

        lang_label = {"en": "English", "tl": "Tagalog", "ceb": "Cebuano"}[lang]
        print(f"\nSynthesizing: {stem} ({lang_label}) → {xtts_lang} phoneme")

        markdown = input_path.read_text(encoding="utf-8")
        plain = preprocess_for_tts(markdown)
        chunks = chunk_text(plain)

        t_start = time.time()
        synthesize_to_mp3(plain, xtts_lang, output_path, tts, speaker=SPEAKER, sample_rate=SAMPLE_RATE)
        elapsed = time.time() - t_start

        size_kb = output_path.stat().st_size // 1024

        # Estimate duration from file size: 128kbps → 16 KB/s
        duration_s = (size_kb * 1024 * 8) / 128_000

        print(f"  ✓ {elapsed:.1f}s  |  {size_kb} KB  |  ~{duration_s:.0f}s audio")

        results.append({
            "stem": stem.replace("PAGASA_", ""),
            "language": lang_label,
            "xtts_lang": xtts_lang,
            "chunks": len(chunks),
            "elapsed_s": round(elapsed, 1),
            "size_kb": size_kb,
            "duration_s": round(duration_s),
            "path": str(output_path),
        })

print(f"\n✓ Done — {len(results)} MP3 files written to {output_dir.absolute()}")

## 4. Results Summary

In [ ]:
print("\nTTS SYNTHESIS SUMMARY")
print("=" * 72)
print(f"{'Bulletin':<28} {'Language':<10} {'Phoneme':<8} {'Chunks':>6} {'Time':>6} {'Size':>7} {'Audio':>7}")
print("-" * 72)

for r in results:
    print(
        f"{r['stem']:<28} {r['language']:<10} {r['xtts_lang']:<8} "
        f"{r['chunks']:>6} {r['elapsed_s']:>5.1f}s {r['size_kb']:>6}KB {r['duration_s']:>6}s"
    )

print("-" * 72)
total_audio = sum(r["duration_s"] for r in results)
print(f"Total audio generated: {total_audio}s ({total_audio/60:.1f} minutes)")
print(f"\n✓ MP3 files ready in: {output_dir.absolute()}")